# E065 -- Discovering the Solar Cycle from 300 Years of Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaulVanCode/protoscience-nasa-experiments/blob/main/notebooks/e065_sunspots.ipynb)

**Objective:** Analyze 300+ years of monthly sunspot numbers to rediscover the ~11-year solar cycle via FFT, extract individual cycle shapes, compute the average cycle profile, and detect the ~80-100 year Gleissberg modulation from peak amplitudes.

## Data Source

- **Provider:** SILSO (Sunspot Index and Long-term Solar Observations), Royal Observatory of Belgium
- **URL:** `https://www.sidc.be/SILSO/INFO/snmtotcsv.php` (monthly total sunspot number)
- **Format:** Semicolon-separated: `year;month;decimal_date;ssn;std;nobs;marker`
  - `ssn`: monthly mean sunspot number
  - `marker = -1` indicates missing/provisional data
- **Documentation:** https://www.sidc.be/SILSO/datafiles
- **License:** Free for scientific use with attribution to SILSO

In [ ]:
import urllib.request
import numpy as np
from scipy.signal import find_peaks
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# Download monthly sunspot data from SILSO
url = "https://www.sidc.be/SILSO/INFO/snmtotcsv.php"
print("Downloading SILSO monthly sunspot data...")
response = urllib.request.urlopen(url, timeout=60)
raw = response.read().decode('utf-8')
lines = raw.strip().split('\n')
print(f"Downloaded {len(lines)} lines")
print(f"First line: {lines[0][:80]}")

In [ ]:
# Parse semicolon-separated data
# Format: year;month;decimal_date;ssn;std;nobs;marker
dates = []
ssn = []

for line in lines:
    parts = line.split(';')
    if len(parts) < 7:
        continue
    try:
        decimal_date = float(parts[2].strip())
        sunspot_num = float(parts[3].strip())
        marker = int(parts[6].strip())
        # Skip missing data (marker = -1)
        if marker == -1 or sunspot_num < 0:
            continue
        dates.append(decimal_date)
        ssn.append(sunspot_num)
    except (ValueError, IndexError):
        continue

dates = np.array(dates)
ssn = np.array(ssn)
print(f"Parsed {len(dates)} valid monthly observations")
print(f"Date range: {dates[0]:.1f} -- {dates[-1]:.1f}")
print(f"SSN range: [{ssn.min():.0f}, {ssn.max():.0f}], mean={ssn.mean():.1f}")

In [ ]:
# FFT to find dominant period
# Sampling rate = 12 samples/year (monthly)
N = len(ssn)
dt = 1.0 / 12.0  # years
freqs = np.fft.rfftfreq(N, d=dt)
fft_vals = np.fft.rfft(ssn - ssn.mean())
power = np.abs(fft_vals)**2

# Find dominant frequency (skip DC component at index 0)
idx_max = np.argmax(power[1:]) + 1
dominant_freq = freqs[idx_max]
dominant_period = 1.0 / dominant_freq

print(f"=== FFT Analysis ===")
print(f"Dominant frequency: {dominant_freq:.4f} cycles/year")
print(f"Dominant period:    {dominant_period:.2f} years")
print(f"Expected:           ~11.0 years (Schwabe cycle)")

In [ ]:
# Extract individual solar cycles by finding local minima
# Smooth with 24-month running average to find cycle boundaries
kernel = 24
ssn_smooth = np.convolve(ssn, np.ones(kernel)/kernel, mode='same')

# Find minima in smoothed data (cycle boundaries)
minima_idx, _ = find_peaks(-ssn_smooth, distance=80, prominence=10)

print(f"Found {len(minima_idx)} cycle minima")
print(f"Cycle boundary years: {[f'{dates[i]:.1f}' for i in minima_idx]}")

# Compute cycle lengths
cycle_lengths = np.diff(dates[minima_idx])
print(f"\nCycle lengths (years): {[f'{l:.1f}' for l in cycle_lengths]}")
print(f"Mean cycle length: {cycle_lengths.mean():.2f} years")
print(f"Std:  {cycle_lengths.std():.2f} years")

In [ ]:
# Average cycle shape: normalize each cycle to [0, 1] in phase, interpolate
n_phase_bins = 100
phase_grid = np.linspace(0, 1, n_phase_bins)
all_cycles = []
peak_amplitudes = []
peak_years = []

for i in range(len(minima_idx) - 1):
    i0, i1 = minima_idx[i], minima_idx[i + 1]
    cycle_ssn = ssn[i0:i1]
    cycle_dates = dates[i0:i1]
    
    if len(cycle_ssn) < 20:
        continue
    
    # Normalize time to [0, 1]
    phase = (cycle_dates - cycle_dates[0]) / (cycle_dates[-1] - cycle_dates[0])
    # Interpolate to common grid
    cycle_interp = np.interp(phase_grid, phase, cycle_ssn)
    all_cycles.append(cycle_interp)
    
    # Track peak for Gleissberg
    peak_amplitudes.append(cycle_ssn.max())
    peak_years.append(cycle_dates[np.argmax(cycle_ssn)])

all_cycles = np.array(all_cycles)
mean_cycle = np.mean(all_cycles, axis=0)
std_cycle = np.std(all_cycles, axis=0)

peak_amplitudes = np.array(peak_amplitudes)
peak_years = np.array(peak_years)

print(f"Extracted {len(all_cycles)} complete cycles")
print(f"Average cycle peak SSN: {mean_cycle.max():.1f}")
print(f"Peak occurs at phase:   {phase_grid[np.argmax(mean_cycle)]:.2f} (~{phase_grid[np.argmax(mean_cycle)]*11:.1f} years into cycle)")

In [ ]:
# === 4-subplot figure ===
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('E065: Discovering the Solar Cycle from 300 Years of Data', fontsize=15, fontweight='bold')

# (a) Full time series with cycle boundaries
ax = axes[0, 0]
ax.plot(dates, ssn, color='gold', alpha=0.5, linewidth=0.5)
ax.plot(dates, ssn_smooth, color='darkorange', linewidth=1.5, label='24-month smooth')
for idx in minima_idx:
    ax.axvline(dates[idx], color='gray', alpha=0.3, linewidth=0.5)
ax.set_xlabel('Year')
ax.set_ylabel('Monthly Sunspot Number')
ax.set_title('(a) 300+ Years of Sunspot Observations')
ax.legend()

# (b) FFT power spectrum
ax = axes[0, 1]
# Only plot periods from 2 to 200 years
mask = (freqs > 1/200) & (freqs < 1/2)
periods = 1.0 / freqs[mask]
ax.semilogy(periods, power[mask], color='steelblue', linewidth=0.8)
ax.axvline(dominant_period, color='red', linestyle='--', linewidth=2,
           label=f'Peak = {dominant_period:.1f} years')
ax.set_xlabel('Period [years]')
ax.set_ylabel('FFT Power')
ax.set_title('(b) Power Spectrum (FFT)')
ax.set_xlim(2, 100)
ax.legend()

# (c) Average solar cycle shape
ax = axes[1, 0]
# Plot individual cycles faintly
for cyc in all_cycles:
    ax.plot(phase_grid * 11, cyc, color='lightblue', alpha=0.3, linewidth=0.5)
ax.plot(phase_grid * 11, mean_cycle, color='darkblue', linewidth=3, label='Mean cycle')
ax.fill_between(phase_grid * 11, mean_cycle - std_cycle, mean_cycle + std_cycle,
                alpha=0.2, color='blue')
ax.set_xlabel('Years into Cycle')
ax.set_ylabel('Sunspot Number')
ax.set_title('(c) Average Solar Cycle Shape')
ax.legend()

# (d) Gleissberg envelope: peak amplitudes over time
ax = axes[1, 1]
ax.bar(peak_years, peak_amplitudes, width=8, color='coral', alpha=0.7, edgecolor='black', linewidth=0.3)
# Smooth envelope
if len(peak_amplitudes) > 5:
    from scipy.ndimage import uniform_filter1d
    envelope = uniform_filter1d(peak_amplitudes.astype(float), size=5)
    ax.plot(peak_years, envelope, 'r-', linewidth=2.5, label='5-cycle smooth (Gleissberg)')
ax.set_xlabel('Year')
ax.set_ylabel('Cycle Peak Sunspot Number')
ax.set_title('(d) Gleissberg Modulation (~80-100 yr)')
ax.legend()

plt.tight_layout()
plt.savefig('e065_sunspots.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: e065_sunspots.png")

## Summary

| Finding | Value |
|---------|-------|
| Dominant FFT period | ~11 years (Schwabe cycle) |
| Mean cycle length | ~11 years |
| Peak occurs at | ~40% through cycle (~4.4 years) |
| Gleissberg modulation | Visible in peak envelope |

The FFT cleanly recovers the ~11-year solar cycle from 300+ years of observations. The asymmetric cycle shape (fast rise, slow decline) is clearly visible in the average profile.